In [ ]:
# â”€â”€ Cell 1: Import all required libraries â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

import os                                    # build cross-platform file paths from parts
import mne                                   # read EDF files and access EEG channel metadata
import numpy as np                           # fast numerical array operations and statistics
from scipy.signal import iirnotch, filtfilt  # design IIR notch filter and apply zero-phase filtering

## EEG Preprocessing Pipeline

This notebook preprocesses raw EEG recordings stored as `.edf` files before they are fed into a deep learning model.

Each recording goes through **three sequential steps**:

| Step | What it does | Why |
|------|-------------|-----|
| **1. 50 Hz Notch filter** | Removes powerline interference using a narrow IIR notch filter applied zero-phase (forward + backward pass). | European mains hum at 50 Hz contaminates almost every EEG recording and must be removed before frequency-domain analysis or model training. |
| **2. Saturation / clipping removal** | Samples that hit the amplifier's voltage rail (hard clip) or stay constant for â‰¥ 5 consecutive samples (flat-line dropout) are marked as NaN and linearly interpolated. | Saturated samples contain no neural information and would corrupt normalisation statistics and model weights. |
| **3. Robust z-score (MAD)** | Each channel is standardised using the median and Median Absolute Deviation (MAD) rather than mean Â± std. | The median and MAD are resistant to large spike artefacts that may remain after clipping removal, making normalisation stable across sessions. |

The function `preprocess_eeg()` below implements this pipeline and returns a **1-D NumPy array** for a single selected EEG channel.

In [ ]:
# â”€â”€ Cell 3: preprocess_eeg() â€” full EEG preprocessing function â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def preprocess_eeg(
    filename,               # name of the EDF file only (e.g. 'subject01.edf'), not the full path
    data_dir,               # path to the folder that contains the EDF file (no hardcoded paths)
    channel_name,           # exact channel label as stored in the EDF header (e.g. 'EEG Fp1')
    notch_freq=50.0,        # powerline frequency to suppress in Hz â€” 50 Hz is the European standard
    notch_quality=30.0,     # Q-factor of the notch filter: higher value = narrower, sharper notch
    sat_threshold=None,     # hard clip level in raw signal units; auto-estimated from data if None
    flat_run_min=5,         # number of consecutive identical samples that flags a flat-line segment
    verbose=True            # print step-by-step progress messages when True
):
    """
    Load one EDF file, extract a single EEG channel, and return a clean,
    normalised 1-D NumPy array ready for model input.

    Parameters
    ----------
    filename      : str          EDF file name (not the full path).
    data_dir      : str          Folder that contains the EDF file.
    channel_name  : str          Channel label as stored in the EDF header.
    notch_freq    : float        Frequency to remove (default 50 Hz).
    notch_quality : float        Notch Q-factor controlling width (default 30).
    sat_threshold : float|None   Clip level in raw signal units; auto if None.
    flat_run_min  : int          Consecutive equal samples before a run is flagged.
    verbose       : bool         Print progress when True.

    Returns
    -------
    signal_norm : np.ndarray, shape (n_samples,), dtype float32
        Preprocessed, normalised EEG signal for the chosen channel.
    """

    # â”€â”€ Step 0: Build the full file path â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    edf_path = os.path.join(data_dir, filename)  # join folder and filename safely on any OS

    # â”€â”€ Step 1: Load the EDF file with MNE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)  # preload=True reads all data into RAM
    sfreq = raw.info['sfreq']                    # sampling frequency in Hz stored in the EDF header

    if verbose:                                  # only print if the caller asked for progress output
        print(f'[1] Loaded   : {filename}')      # confirm which file was opened
        print(f'    Channels : {len(raw.ch_names)}  |  Fs: {sfreq} Hz  |  Duration: {raw.times[-1]:.1f} s')
        print(f'    Available channels: {raw.ch_names}')  # list all channel names so the caller can verify

    # â”€â”€ Step 2: Extract the requested single EEG channel â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if channel_name not in raw.ch_names:         # guard: raise a clear error if the channel does not exist
        raise ValueError(
            f"Channel '{channel_name}' not found. "
            f"Available channels: {raw.ch_names}"
        )

    raw_ch = raw.copy().pick_channels([channel_name])          # keep only the requested channel
    signal = raw_ch.get_data().squeeze().astype(np.float64)    # shape (n_samples,) â€” squeeze removes the channel axis

    if verbose:
        print(f"[2] Extracted channel '{channel_name}'  |  {len(signal)} samples")  # confirm extraction

    # â”€â”€ Step 3: 50 Hz notch filter â€” remove powerline interference â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    b, a = iirnotch(
        w0=notch_freq / (sfreq / 2),  # normalise notch frequency to the Nyquist frequency (must be in [0, 1])
        Q=notch_quality               # Q-factor: Q=30 gives a notch ~1.7 Hz wide at 50 Hz
    )
    signal = filtfilt(b, a, signal)   # zero-phase filtering: forward then backward pass cancels phase distortion

    if verbose:
        print(f'[3] Notch filter applied at {notch_freq} Hz  (Q={notch_quality})')  # confirm filter step

    # â”€â”€ Step 4a: Hard-clip saturation detection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if sat_threshold is None:                    # auto-estimate threshold when not provided by caller
        sat_threshold = np.percentile(np.abs(signal), 99.9)  # 99.9th percentile is high enough to spare genuine EEG peaks

    signal = signal.astype(np.float64)           # ensure float64 so we can assign NaN (int arrays cannot hold NaN)
    signal[np.abs(signal) >= sat_threshold] = np.nan  # replace every saturated sample with NaN

    if verbose:
        n_sat = int(np.sum(np.isnan(signal)))    # count how many samples were clipped
        print(f'[4a] Saturation threshold: {sat_threshold:.4e}  |  {n_sat} samples removed ({100*n_sat/len(signal):.2f}%)')

    # â”€â”€ Step 4b: Flat-line (soft-clip / dropout) detection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    equal = np.concatenate(([False], np.diff(signal) == 0, [False]))  # True where adjacent samples are identical; padded for edge safety
    run_starts = np.where(~equal[:-1] &  equal[1:])[0]               # index where each flat run begins
    run_ends   = np.where( equal[:-1] & ~equal[1:])[0] + 1           # index just after each flat run ends

    n_flat = 0                                   # accumulate total flat samples removed for reporting
    for s, e in zip(run_starts, run_ends):       # iterate over every detected flat-line segment
        if (e - s) >= flat_run_min:              # only flag runs long enough to be real artefacts
            signal[s:e] = np.nan                 # mark the entire flat segment as missing
            n_flat += (e - s)                    # add segment length to the running count

    if verbose:
        print(f'[4b] Flat-line segments removed: {n_flat} samples')  # report flat-line removal

    # â”€â”€ Step 4c: Linear interpolation over all NaN gaps â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    nan_mask = np.isnan(signal)                  # Boolean mask: True wherever a sample was removed
    good_idx = np.where(~nan_mask)[0]            # indices of all remaining valid (non-NaN) samples

    if len(good_idx) > 1:                        # need at least 2 good points to interpolate
        signal = np.interp(
            np.arange(len(signal)),              # query every sample index
            good_idx,                            # x-coordinates of known-good samples
            signal[good_idx]                     # y-values of known-good samples
        )
    elif len(good_idx) == 0:                     # channel is completely bad â€” cannot recover
        signal[:] = 0.0                          # zero the channel rather than leaving NaN downstream
        if verbose:
            print(f"    Warning: channel '{channel_name}' had no valid samples and was zeroed.")  # alert the caller

    if verbose:
        print(f'    Interpolation done  |  NaN remaining: {int(np.sum(np.isnan(signal)))}')  # confirm no NaN left

    # â”€â”€ Step 5: Robust z-score normalisation (MAD) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Formula:  z = (x âˆ’ median(x)) / (1.4826 Ã— MAD(x))
    # The 1.4826 factor makes MAD equal to std for Gaussian data,
    # so the output z-scores are on the same scale as classical standardisation.
    med = np.median(signal)                      # robust location estimate â€” not affected by residual spikes
    mad = np.median(np.abs(signal - med))        # Median Absolute Deviation â€” robust scale estimate
    mad_safe = mad if mad != 0.0 else 1.0        # guard against dead/constant channels where MAD == 0
    signal_norm = (signal - med) / (1.4826 * mad_safe)  # apply the robust z-score formula element-wise

    if verbose:
        print(f'[5] Robust z-score applied  |  median={med:.4e}  MAD={mad:.4e}')  # report normalisation stats
        print(f'    Output shape : {signal_norm.shape}  |  range [{signal_norm.min():.2f}, {signal_norm.max():.2f}]')

    return signal_norm.astype(np.float32)        # cast to float32 to halve memory â€” sufficient for deep learning

In [ ]:
# ── Cell 5: preprocess_folder() — batch process a folder of EDF files ──────────

def preprocess_folder(
    data_dir,           # path to the folder containing all EDF files
    channel_name,       # EEG channel label to extract from every file (must be consistent across files)
    out_dir=None,       # folder where .npy files are saved; defaults to data_dir if not specified
    notch_freq=50.0,    # powerline frequency to suppress in Hz — passed through to preprocess_eeg()
    notch_quality=30.0, # Q-factor of the notch filter — passed through to preprocess_eeg()
    sat_threshold=None, # clipping threshold — passed through to preprocess_eeg(); None = auto per file
    flat_run_min=5,     # minimum flat-line run length in samples — passed through to preprocess_eeg()
    verbose=True        # print per-file progress when True
):
    """
    Batch-preprocess every EDF file in data_dir.

    Each file is processed independently by preprocess_eeg() and its output
    (a 1-D float32 NumPy array) is saved as a separate .npy file.  Signals are
    NOT stacked because recordings may have different lengths.

    Parameters
    ----------
    data_dir      : str          Folder containing the EDF files.
    channel_name  : str          EEG channel label to extract from every file.
    out_dir       : str|None     Destination folder for .npy files (default: data_dir).
    notch_freq    : float        Notch frequency in Hz (default 50).
    notch_quality : float        Notch Q-factor (default 30).
    sat_threshold : float|None   Clipping threshold; auto-detected per file if None.
    flat_run_min  : int          Minimum flat-line run length to flag (default 5).
    verbose       : bool         Print progress per file when True.

    Returns
    -------
    summary : dict
        Keys are EDF file names; values are dicts with keys:
            'npy_path'  – absolute path of the saved .npy file, or
            'error'     – error message string if the file failed.
    """

    # ── Resolve output directory ──────────────────────────────────────────────
    if out_dir is None:                              # use data_dir as output location if not specified
        out_dir = data_dir
    os.makedirs(out_dir, exist_ok=True)             # create the output folder if it does not already exist

    # ── Collect all EDF files in the folder ───────────────────────────────────
    all_files = sorted(                             # sort for reproducible processing order
        f for f in os.listdir(data_dir)             # list every entry in the directory
        if f.lower().endswith('.edf')               # keep only files with an .edf extension
    )

    if len(all_files) == 0:                         # stop early if the folder contains no EDF files
        raise FileNotFoundError(f"No .edf files found in: {data_dir}")

    if verbose:
        print(f"Found {len(all_files)} EDF file(s) in: {data_dir}")  # report how many files will be processed
        print(f"Output folder : {out_dir}\n")

    summary = {}                                    # dict to collect per-file outcomes for the caller

    # ── Process each file ─────────────────────────────────────────────────────
    for idx, filename in enumerate(all_files):      # iterate over every EDF file in sorted order

        if verbose:
            print(f"[{idx+1}/{len(all_files)}] Processing: {filename}")  # show progress counter

        try:
            # Call the single-file preprocessing function.
            # verbose=False suppresses per-step detail to keep console output readable across 102 files.
            signal = preprocess_eeg(
                filename=filename,            # EDF file name (folder is provided separately)
                data_dir=data_dir,            # folder containing the EDF file
                channel_name=channel_name,    # channel to extract — same label expected in every file
                notch_freq=notch_freq,        # 50 Hz notch frequency
                notch_quality=notch_quality,  # notch Q-factor
                sat_threshold=sat_threshold,  # clipping threshold (None = auto per file)
                flat_run_min=flat_run_min,    # minimum flat-line run length
                verbose=False                 # suppress per-step output during batch processing
            )

            # Build the output .npy file path: same stem as the EDF, with .npy extension.
            stem     = os.path.splitext(filename)[0]           # strip the .edf extension from the file name
            npy_name = stem + '.npy'                           # e.g. subject01_session1.npy
            npy_path = os.path.join(out_dir, npy_name)         # full path to the output file

            # Save the 1-D NumPy array to disk in NumPy's native binary format.
            # np.save writes a compact .npy file that can be reloaded with np.load.
            np.save(npy_path, signal)                          # saves shape, dtype, and data — no extra dependencies

            summary[filename] = {'npy_path': npy_path}         # record success with the saved file path

            if verbose:
                print(f"    Saved -> {npy_path}  |  shape: {signal.shape}  dtype: {signal.dtype}")  # confirm save

        except Exception as exc:                               # catch any error so one bad file does not abort the batch
            summary[filename] = {'error': str(exc)}            # record the error message for this file
            print(f"    WARNING: {filename} failed — {exc}")   # always print failures regardless of verbose flag

    # ── Print final summary ───────────────────────────────────────────────────
    ok     = sum(1 for v in summary.values() if 'npy_path' in v)  # count successfully processed files
    failed = sum(1 for v in summary.values() if 'error'    in v)  # count failed files

    print(f"\nBatch complete: {ok} succeeded, {failed} failed out of {len(all_files)} files.")

    return summary  # return the per-file outcome dict so the caller can inspect or log it


In [ ]:
# â”€â”€ Cell 4: Example â€” call preprocess_eeg() with placeholder values â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

data_dir     = '/path/to/your/edf/folder'   # replace with the actual folder containing your EDF files
filename     = 'subject01_session1.edf'     # replace with the actual EDF file name
channel_name = 'EEG Fp1'                    # replace with the channel label stored in your EDF header

clean_signal = preprocess_eeg(              # run the full preprocessing pipeline on the specified file
    filename=filename,                      # EDF file name (not the full path)
    data_dir=data_dir,                      # folder that contains the EDF file
    channel_name=channel_name,             # which EEG channel to extract and clean
    notch_freq=50.0,                        # suppress 50 Hz powerline interference
    notch_quality=30.0,                     # notch width: Q=30 keeps the notch narrow and precise
    sat_threshold=None,                     # None = auto-detect the clipping threshold from the data
    flat_run_min=5,                         # flag any run of 5 or more consecutive identical samples
    verbose=True                            # print each preprocessing step to the console
)

print(f'\nOutput type  : {type(clean_signal)}')                                 # confirm it is a NumPy array
print(f'Output dtype : {clean_signal.dtype}')                                    # confirm float32
print(f'Output shape : {clean_signal.shape}')                                    # confirm 1-D array (n_samples,)
print(f'Value range  : [{clean_signal.min():.2f}, {clean_signal.max():.2f}]')   # inspect the z-score value range